In [1]:
import pandas as pd
import numpy as np
from data_names import huc
import os

In [2]:
#change this to your directory
base_dir = ""

daymet_mean_forcing_dir = f"{base_dir}/basin_mean_forcing/daymet"
basin_metadata_dir = f"{base_dir}/basin_metadata"
usgs_streamflow_dir = f"{base_dir}/usgs_streamflow"


In [ ]:
#clean each individual directory and joinal

#clean daymet mean forcing - Alex
#input:  daymet_dir = path to basin_mean_forcing/daymet/
#        hucs = region folders to read, e.g. ("17",)
#output: DataFrame, one row per gauge per day
#        gauge_id (text, from the filename), date (one datetime column), weather columns
#note:   each file has 3 header lines before the column names
def clean_daymet(daymet_dir, hucs=("17",)):
    pass


In [ ]:
#clean basin_metadata - JJ
#input:  metadata_dir = path to basin_metadata/
#        hucs = region codes to keep, e.g. ("17",) or ("17", "18")
#output: DataFrame, one row per gauge, 13 columns
#steps:
#1. gauge_information.txt -> huc, gauge_id, name, lat, lon, drainage area; keep only hucs
#2. basin_physical_characteristics.txt -> area, elevation, slope, forest fraction
#3. basin_annual_hydrometeorology_characteristics_daymet.txt -> long-term precip, PET, temp
#4. rename ids to gauge_id (kept as text for leading zeros) and merge all 3 on gauge_id
def clean_basin_metadata(metadata_dir, hucs=("17",)):
    #tab-separated, and gauge names contain spaces
    gauges = pd.read_csv(
        f"{metadata_dir}/gauge_information.txt",
        sep="\t",
        skiprows=1,
        header=None,
        names=["huc", "gauge_id", "gauge_name", "lat", "lon", "drainage_area_km2"],
        dtype={"huc": str, "gauge_id": str},
    )
    gauges["gauge_name"] = gauges["gauge_name"].str.strip()
    gauges = gauges[gauges["huc"].isin(hucs)]

    physical = pd.read_csv(
        f"{metadata_dir}/basin_physical_characteristics.txt",
        sep=r"\s+",
        dtype={"BASIN_ID": str},
    )
    physical = physical.drop(columns="BASIN_HUC").rename(columns={
        "BASIN_ID": "gauge_id",
        "Size(km2)": "area_km2",
        "Elevation(m)": "elevation_m",
        "Slope(m_km-1)": "slope_m_per_km",
        "Frac_Forest(percent)": "frac_forest",
    })

    #long-term climate averages; drop Runoff since it comes from streamflow (our label)
    climate = pd.read_csv(
        f"{metadata_dir}/basin_annual_hydrometeorology_characteristics_daymet.txt",
        sep=r"\s+",
        dtype={"BASIN_ID": str},
    )
    climate = climate.drop(columns=["BASIN_HUC", "Runoff(mm_day-1)"]).rename(columns={
        "BASIN_ID": "gauge_id",
        "Precip(mm_day-1)": "mean_precip_mm_day",
        "PET(mm_day-1)": "mean_pet_mm_day",
        "Temp(oC)": "mean_temp_c",
    })

    return gauges.merge(physical, on="gauge_id").merge(climate, on="gauge_id")

In [3]:
#clean usgs streamflow - Franzi
#input:streamflow_dir = path to usgs_streamflow/
#        hucs = region folders to read, e.g. ("17",)
#output: DataFrame, one row per gauge per day
#        gauge_id (text), date (one datetime column), flow
#note: missing flow is -999/-999.0 (flag M) and must be removed
#Franzi note:flow is measured in cubic feet/second so we might want/need to convert units later 
def clean_usgs_streamflow(streamflow_dir, hucs=("17",)):
    total_df = []
    
    #Loop through region by huc code
    for huc in hucs:
        huc_path = os.path.join(streamflow_dir, huc)
        
        for filesname in os.listdir(huc_path):
            if not filesname.endswith(".txt"):
                continue
            
            files = os.path.join(huc_path, filesname)
            
            gauge_id = str(filesname.split('_')[0]).zfill(8)
            df = pd.read_csv(files, sep=r'\s+', header=None)
            df = df[[1,2,3,4]]
            df.columns = ['year', 'month', 'day', 'flow_cfs']
            df['date'] = pd.to_datetime(df[['year', 'month', 'day']])
            
            df['gauge_id'] = gauge_id
            
            #replace missing valueswith nan
            df['flow_cfs'] = df['flow_cfs'].replace([-999, -999.0], np.nan)
            clean_df = df[['gauge_id', 'date', 'flow_cfs']]
            
            #append
            total_df.append(clean_df)
        
    return pd.concat(total_df, ignore_index = True)

In [4]:
#test1
test_streamflow_df = clean_usgs_streamflow(usgs_streamflow_dir, hucs=("17",))

print(test_streamflow_df.dtypes)

print(f'{test_streamflow_df.head()}')
test_streamflow_df

gauge_id               str
date        datetime64[us]
flow_cfs           float64
dtype: object
   gauge_id       date  flow_cfs
0  10396000 1980-01-01      54.0
1  10396000 1980-01-02      50.0
2  10396000 1980-01-03      50.0
3  10396000 1980-01-04      50.0
4  10396000 1980-01-05      75.0


,gauge_id,date,flow_cfs
0,10396000,1980-01-01,54.0
1,10396000,1980-01-02,50.0
2,10396000,1980-01-03,50.0
3,10396000,1980-01-04,50.0
4,10396000,1980-01-05,75.0
...,...,...,...
1135845,14400000,2014-12-27,3800.0
1135846,14400000,2014-12-28,3220.0
1135847,14400000,2014-12-29,2810.0
1135848,14400000,2014-12-30,2460.0


In [5]:
#test2
nan_rows = test_streamflow_df[test_streamflow_df['flow_cfs'].isna()]
print(nan_rows.head())

       gauge_id       date  flow_cfs
12760  10396000 2014-12-08       NaN
12761  10396000 2014-12-09       NaN
12762  10396000 2014-12-10       NaN
12763  10396000 2014-12-11       NaN
12764  10396000 2014-12-12       NaN


In [ ]:
#join all three sources - one row per gauge per day
#input:  streamflow, daymet = DataFrames with gauge_id (text) and date
#        metadata = DataFrame with one row per gauge_id
#output: DataFrame, one row per gauge per day, with flow + weather + basin facts
#steps:
#1. merge streamflow and daymet on gauge_id + date (both are daily)
#2. merge metadata on gauge_id only, so basin facts repeat on every day
#   (many_to_one errors if a gauge appears twice in metadata)
def join_data(streamflow, daymet, metadata):
    daily = streamflow.merge(daymet, on=["gauge_id", "date"])
    return daily.merge(metadata, on="gauge_id", validate="many_to_one")

In [ ]:
#save the final table
#input:  df = joined DataFrame from join_data, path = output file, e.g. "pnw_dataset.csv"
#output: writes the CSV (no row index), returns nothing
#note:   read it back with pd.read_csv(path, dtype={"gauge_id": str}) to keep leading zeros
def generate_csv(df, path):
    df.to_csv(path, index=False)

In [ ]:
#run the whole pipeline - change hucs here to pick regions for every step
#until clean_daymet and clean_usgs_streamflow return DataFrames, this only shows metadata
hucs = ("17",)

metadata = clean_basin_metadata(basin_metadata_dir, hucs)
daymet = clean_daymet(daymet_mean_forcing_dir, hucs)
streamflow = clean_usgs_streamflow(usgs_streamflow_dir, hucs)

if daymet is None or streamflow is None:
    print("daymet or streamflow not finished yet - showing metadata only")
    display(metadata.head())
else:
    dataset = join_data(streamflow, daymet, metadata)
    generate_csv(dataset, "pnw_dataset.csv")
    print(dataset.shape)
    display(dataset.head())